In [1]:
cd lib/

/home/lowen/Documents/github/physics-of-agents/lib


In [2]:
from pathlib import Path

files = sorted(Path("../data/models").glob("*/*/*.json"))

print("Number of files:", len(files))
print(*files[:10], sep="\n")

Number of files: 9614
../data/models/gemini-3.5-flash-lite/subjective_energy/manifest.json
../data/models/gemini-3.5-flash-lite/subjective_energy/political_stance_0__J0__rep00.json
../data/models/gemini-3.5-flash-lite/subjective_energy/political_stance_156__J0__rep00.json
../data/models/gemma-3n-e4b-it/objective_energy/manifest.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep00.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep01.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep02.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep03.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep00.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep01.json


In [3]:
from pathlib import Path

files = list(Path("../data/models").glob("*/*/*.json"))

total = sum(f.stat().st_size for f in files)

print("files:", len(files))
print("total GB:", total / 1024**3)

by_model = {}

for f in files:
    model = f.parts[-3]
    by_model.setdefault(model, 0)
    by_model[model] += f.stat().st_size

for model, size in sorted(by_model.items(), key=lambda x: -x[1]):
    print(f"{model:50s} {size / 1024**2:10.1f} MB")

files: 9614
total GB: 3.182823774404824
meta-llama-3-8b-instruct                                917.5 MB
qwen3.5-9b                                              854.3 MB
gpt-4o-mini                                             780.4 MB
gemma-3n-e4b-it                                         707.0 MB
gemini-3.5-flash-lite                                     0.0 MB
mock                                                      0.0 MB


In [4]:
from pathlib import Path
import json
import time

files = sorted(Path("../data/models").glob("*/*/*.json"))

print("files:", len(files))

for f in files:
    if "gemini-3.5-flash-lite" in str(f):
        print("\nGEMINI:", f)
        print("size:", f.stat().st_size / 1024, "KB")

        with open(f) as fh:
            x = json.load(fh)

        print("top-level keys:")
        print(list(x.keys()))

files: 9614

GEMINI: ../data/models/gemini-3.5-flash-lite/subjective_energy/manifest.json
size: 1.1259765625 KB
top-level keys:
['meta', 'replicas']

GEMINI: ../data/models/gemini-3.5-flash-lite/subjective_energy/political_stance_0__J0__rep00.json
size: 8.599609375 KB
top-level keys:
['meta', 'personas', 'statement', 'mode', 'choices', 'J', 'spins_history', 'spins_raw_history', 'messages_history']

GEMINI: ../data/models/gemini-3.5-flash-lite/subjective_energy/political_stance_156__J0__rep00.json
size: 8.44140625 KB
top-level keys:
['meta', 'personas', 'statement', 'mode', 'choices', 'J', 'spins_history', 'spins_raw_history', 'messages_history']


In [5]:
for f in files:
    if "gemma-3n-e4b-it" in str(f):
        print("\nEXISTING:", f)

        with open(f) as fh:
            x = json.load(fh)

        print("top-level keys:")
        print(list(x.keys()))

        break


EXISTING: ../data/models/gemma-3n-e4b-it/objective_energy/manifest.json
top-level keys:
['meta', 'replicas']


In [7]:
import json, glob
from pathlib import Path
import pandas as pd

records = []
for f in sorted(Path("../data/models").glob("*/*/*.json")):
    if f.name == "manifest.json":
        continue
    rec = json.load(open(f))
    rec["model_dir"] = f.parts[-3]   # e.g. gemma-3n-e4b-it
    rec["energy_mode"] = f.parts[-2] # objective_energy / subjective_energy
    rec["file"] = f.name
    records.append(rec)

# df = pd.json_normalize(records)

print("before normalize", flush=True)

df = pd.json_normalize(records)

print("after normalize", flush=True)
print("shape:", df.shape, flush=True)
print("memory:", df.memory_usage(deep=True).sum() / 1024**3, "GB", flush=True)

parts = df["file"].str.extract(r"(?P<question>.+)__(?P<graph>[^_]+)__rep(?P<rep>\d+)\.json")
df = df.join(parts)
df["replica"] = df["rep"].astype(int)

df["model"] = df["meta.model"]

# load them and add to the df here
obj = pd.concat([
    pd.read_json("../data/obj/train.jsonl", lines=True),
    pd.read_json("../data/obj/test.jsonl", lines=True),
])
ground_truth = obj.set_index("qid")["answer"]              # "A" or "B"
lean = json.load(open("../data/subj/lean.json"))["lean"]   # qid -> left/right/ambiguous

df["ground_truth"] = df["question"].map(ground_truth)      # NaN on subjective rows
df["political_lean"] = df["question"].map(lean)            # NaN on objective rows

# sanity check: every row got exactly one of the two labels
assert (df["ground_truth"].notna() ^ df["political_lean"].notna()).all()
cols = ['model', 'mode', 'statement', 'J', 'spins_history', 'spins_raw_history',
        'replica', 'ground_truth', 'political_lean']
df[cols].to_json("../data/clean_runs.json", orient="records")

# from datasets import Dataset
# ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
# ds.push_to_hub("physics-of-agents/agent-opinions", private=False)

before normalize
after normalize
shape: (9604, 43)
memory: 0.024901416152715683 GB


In [8]:
from pathlib import Path

p = Path("../data/clean_runs.json")
print(p.resolve())
print(p.exists())
print(p.stat().st_size if p.exists() else None)

/home/lowen/Documents/github/physics-of-agents/data/clean_runs.json
True
73215754
